# 사출 불량 예측: 데이터 점검과 불균형 처리 비교

## 왜 이 실험을 하는가
RF 튜닝의 제한적인 오탐 감소 이후, 준지도 DNN에서 CN7 AP가 1회차 .4919에서 22회차 .2900으로 낮아지고 RG3는 최종 불량 8건을 모두 놓쳤다. 모델을 더 오래 학습하기보다 데이터와 불균형 처리 효과를 확인한다.

**이번 실험은 성능 개선을 보장하지 않는다.** 작성 시점에는 학습을 실행하지 않았다. 기존에 결과를 확인한 11월 데이터를 다시 보고하므로 미사용 최종 시험이 아닌 후속 탐색이다.

- 원본 `labeled_data.csv` 사용. 가공 CSV는 원본과 관계를 점검하는 용도이며 합쳐 학습하지 않는다.
- 제품명(LH/RH 포함)을 입력에 유지한다. 검사 이후 변수 Reason, 정답, 시간, ID, 생산 지시 코드는 예측 입력에서 제외한다.
- 동일한 RF 구조에서 기본·기존 가중치·오버샘플링 두 비율을 비교한다. DNN·미라벨 반복 학습·SMOTE는 추가하지 않는다.
- 모델 설정은 검증 평균 AP로 비교하되, 기존 RF보다 평균 AP가 높고 3개 검증 구간 중 2개 이상에서 개선될 때만 변경한다.
- 검증 OOF 예측으로 검사 대상 10% 이내 F1 최대 임계값을 정한다. 동률이면 FP가 적고 임계값이 높은 것을 선택한다.
- 11월은 선택된 모델과 기준 RF만 같은 기록으로 보고한다. 그 결과로 재선택하지 않는다.

**전체 실행은 최대 RF 학습 14회(4조건×3구간 + 최종 최대2회), 각 200개 트리다.** GPU나 별도 유료 런타임은 필요하지 않다. 실행 시간은 실제 환경에 따라 다르며 첫 비교 셀에서 경과 시간을 출력한다.

노트북 번호 1~4는 설정·데이터 점검이다. **5~6번은 사용자가 실행하는 모델 학습**, 7~8번은 평가·결과 저장이다.

## 1. Drive 연결과 경로 설정

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, platform, zipfile
from datetime import datetime
try:
    from google.colab import drive
except ImportError:
    IN_COLAB=False
else:
    IN_COLAB=True
    drive.mount('/content/drive')

# CSV 파일이 들어 있는 폴더. 실제 위치에 맞게 수정한다.
DATA_DIR = Path(os.environ.get('MOLDING_DATA_DIR',
    '/content/drive/MyDrive/사출 공정 불량 예측 및 검사 기준 분석/dataset'))
OUTPUT_ROOT = Path(os.environ.get('MOLDING_IMBALANCE_OUTPUT',
    '/content/drive/MyDrive/molding_imbalance_results'))
assert (DATA_DIR/'labeled_data.csv').is_file(), 'DATA_DIR를 labeled_data.csv가 있는 폴더로 수정하세요.'
RESULT_DIR=OUTPUT_ROOT/datetime.now().strftime('run_%Y%m%d_%H%M%S')
RESULT_DIR.mkdir(parents=True,exist_ok=True)
print('결과 저장 폴더:',RESULT_DIR)

## 2. 라이브러리와 사전 고정 설정

In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix

SEED=42
BUDGET=.10
BASELINE='rf_baseline'
SPECS={
    'rf_baseline': {'class_weight':'balanced_subsample','oversample_ratio':None},
    'rf_plain': {'class_weight':None,'oversample_ratio':None},
    'rf_ros_025': {'class_weight':None,'oversample_ratio':.25},
    'rf_ros_100': {'class_weight':None,'oversample_ratio':1.0},
}
# 학습은 시작일 이전, 검증은 시작일 이상·종료일 미만이다.
FOLDS=[('D1','2020-10-22','2020-10-23'),
       ('D2','2020-10-23','2020-10-24'),
       ('D3','2020-10-27','2020-11-01')]
PROTOCOL={'seed':SEED,'budget':BUDGET,'folds':FOLDS,'candidates':SPECS,
    'rf':{'n_estimators':200,'max_depth':None,'min_samples_leaf':5},
    'selection':'mean_AP above baseline and improvements on at least two folds; otherwise baseline',
    'threshold':'pooled temporal OOF, F1 maximum within 10% inspections',
    'report_start':'2020-11-01','report_end_exclusive':'2020-11-07',
    'status':'exploratory reuse of previously observed evaluation period'}
(RESULT_DIR/'protocol.json').write_text(json.dumps(PROTOCOL,ensure_ascii=False,indent=2),encoding='utf-8')
(RESULT_DIR/'environment.json').write_text(json.dumps({
    'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
    'sklearn':sklearn.__version__},indent=2),encoding='utf-8')
print('설정:',len(SPECS),'조건 ×',len(FOLDS),'검증 구간. GPU 사용 없음.')

## 3. 데이터 관계·라벨·동일 공정값 확인

완전히 동일한 원본 행만 제거한다. 서로 다른 생산 건의 공정값이 같다고 해서 제거하지 않는다. ID/생산 사건 키가 중복되거나 라벨이 충돌하면 자동 합치기 대신 실행을 중단한다.

가공 파일에서 공정값이 같고 라벨이 다른 것은 입력 정보만으로 구분할 수 없는 사례가 있다는 뜻이다. **같은 생산 사건이거나 라벨 오류라고 단정하지 않는다.** 파일에 제품 LH/RH 정보가 없을 수 있으므로 원본의 제품 구분을 유지한다.

In [ ]:
def read_csv(name):
    d=pd.read_csv(DATA_DIR/name)
    return d.loc[:,~d.columns.str.startswith('Unnamed:')]

audit=[];manifest=[]
for name in ['labeled_data.csv','moldset_labeled.csv','supervised_label_cn7.csv',
             'moldset_labeled_cn7.csv','moldset_labeled_rg3.csv']:
    if not (DATA_DIR/name).exists(): continue
    d=read_csv(name)
    y=d.PassOrFail.astype(str).map({'Y':0,'N':1,'0':0,'1':1,'0.0':0,'1.0':1})
    assert y.notna().all(),f'{name}: 알 수 없는 라벨'
    row={'file':name,'rows':len(d),'bad_rows':int(y.sum()),
         'distinct_full_rows':len(d.drop_duplicates())}
    if name in ['moldset_labeled_cn7.csv','moldset_labeled_rg3.csv']:
        f=d.drop(columns='PassOrFail')
        conflicting=d.groupby(list(f.columns),dropna=False).PassOrFail.transform('nunique').gt(1)
        row['bad_with_identical_normal_features']=int(y.loc[conflicting].sum())
    audit.append(row)
    manifest.append({'file':name,'sha256':hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()})

raw=read_csv('labeled_data.csv')
clean=raw.drop_duplicates().copy()
assert not clean['_id'].duplicated().any(), '동일 ID의 서로 다른 기록을 먼저 검토해야 합니다.'
if (DATA_DIR/'moldset_labeled.csv').exists():
    m=read_csv('moldset_labeled.csv')
    joined=m.merge(clean[['_id','PassOrFail']],on='_id',how='left',suffixes=('_processed','_raw'),validate='many_to_one')
    extra=int((~m['_id'].isin(clean['_id'])).sum())
    matched=joined.PassOrFail_raw.notna()
    disagreement=int((joined.loc[matched,'PassOrFail_raw'].map({'Y':0,'N':1}).astype(int)
        !=joined.loc[matched,'PassOrFail_processed'].astype(int)).sum())
    print('moldset_labeled에서 원본에 없는 ID:',extra,'동일 ID의 라벨 불일치:',disagreement)
    assert disagreement==0, '라벨 충돌 검토 필요'

allowed=["CN7 W/S SIDE MLD'G LH","CN7 W/S SIDE MLD'G RH",
         "RG3 MOLD'G W/SHLD, LH","RG3 MOLD'G W/SHLD, RH"]
frame=clean.loc[clean.EQUIP_CD.eq('S14') & clean.PART_NAME.isin(allowed)].copy()
frame['time']=pd.to_datetime(frame.TimeStamp,errors='raise')
frame['defect']=frame.PassOrFail.map({'Y':0,'N':1}).astype(int)
assert not frame.duplicated(['TimeStamp','EQUIP_CD','PART_NAME']).any(), '동일 생산 사건 키를 확인하세요.'
frame=frame.sort_values(['time','_id']).reset_index(drop=True)
meta={'_id','TimeStamp','time','defect','PassOrFail','Reason','PART_FACT_PLAN_DATE',
      'PART_FACT_SERIAL','PART_NAME','EQUIP_CD','EQUIP_NAME','PART_NO'}
NUMERIC=[c for c in frame.columns if c not in meta]
frame[NUMERIC]=frame[NUMERIC].apply(pd.to_numeric,errors='raise')
frame[NUMERIC]=frame[NUMERIC].replace([np.inf,-np.inf],np.nan)
representation=[]
for include_part in [False,True]:
    key=NUMERIC+(['PART_NAME'] if include_part else [])
    conflict=frame.groupby(key,dropna=False).defect.transform('nunique').gt(1)
    representation.append({'include_part':include_part,
        'bad_with_identical_normal_features':int(frame.loc[conflict,'defect'].sum())})
display(pd.DataFrame(representation))
pd.DataFrame(representation).to_csv(RESULT_DIR/'representation_audit.csv',index=False)
display(pd.DataFrame(audit))
print('분석 대상:',len(frame),'불량:',int(frame.defect.sum()))
period=np.where(frame.time.lt('2020-11-01'),'before_nov','november')
display(frame.assign(period=period).groupby(['period','PART_NAME']).defect.agg(['size','sum']))
pd.DataFrame(audit).to_csv(RESULT_DIR/'data_audit.csv',index=False)
pd.DataFrame(manifest).to_csv(RESULT_DIR/'input_manifest.csv',index=False)

## 4. 시간순 비교 구간 확인 — 아직 모델 학습 없음

| 구간 | 학습 | 검증 |
|---|---|---|
| D1 | 10/22 이전 | 10/22 |
| D2 | 10/23 이전 | 10/23 |
| D3 | 10/27 이전 | 10/27~10/31 |

검증 세 구간은 서로 겹치지 않는다. 같은 생산 지시 묶음이 각 학습·검증 경계에 걸리지 않는지도 검사한다. D1은 RG3 불량을 학습에서 보지 못한 상태로 평가한다. D3는 CN7 위주다. 검증 제품 구성이 다르므로 평균 점수뿐 아니라 각 구간을 보고한다.

기존 튜닝과 달리 D3까지 선택에 사용하고, 최종 학습에는 11월 이전 전체를 사용한다. 따라서 **기존에 적어둔 AP .1195와 직접 개선율을 계산하지 않고**, 같은 새 조건으로 재학습한 기준 RF와 비교한다. 11월 평가 지표는 설정·임계값 선택에 사용하지 않는다.

In [ ]:
def production_keys(d):
    return set(d[['PART_FACT_PLAN_DATE','PART_FACT_SERIAL','EQUIP_CD']].astype(str).agg('|'.join,axis=1))
def verify_pair(a,b):
    assert a.time.max()<b.time.min()
    assert set(a['_id']).isdisjoint(b['_id'])
    assert production_keys(a).isdisjoint(production_keys(b))
    assert a.defect.nunique()==2 and b.defect.nunique()==2
split_table=[];fold_data=[];seen=set()
for name,start,end in FOLDS:
    a=frame.loc[frame.time.lt(start)].copy()
    b=frame.loc[frame.time.ge(start)&frame.time.lt(end)].copy()
    verify_pair(a,b)
    assert seen.isdisjoint(b['_id']);seen.update(b['_id'])
    fold_data.append((name,a,b))
    for role,d in [('train',a),('validation',b)]:
        split_table.append({'fold':name,'role':role,'rows':len(d),'bad':int(d.defect.sum())})
refit=frame.loc[frame.time.lt('2020-11-01')].copy()
report=frame.loc[frame.time.ge('2020-11-01') & frame.time.lt('2020-11-07')].copy()
verify_pair(refit,report)
split_table.extend([{'fold':'final','role':'train','rows':len(refit),'bad':int(refit.defect.sum())},
                    {'fold':'final','role':'report','rows':len(report),'bad':int(report.defect.sum())}])
display(pd.DataFrame(split_table))
pd.DataFrame(split_table).to_csv(RESULT_DIR/'split_summary.csv',index=False)

## 5. 사용자 실행: RF 4조건 비교 — 총 12회 학습

트리 200개, 최대 깊이 제한 없음, 최소 잎 표본 5, seed 42를 모두 동일하게 둔다.

- `rf_baseline`: 기존 선택 RF의 balanced_subsample 가중치.
- `rf_plain`: 가중치·오버샘플링 없음.
- `rf_ros_025`: 학습 불량/정상 비율을 최대 .25까지 복제하여 증가.
- `rf_ros_100`: 학습 불량/정상 비율을 최대 1:1까지 복제하여 증가.

가공·대체·인코딩은 원래 학습 구간으로만 맞춘 다음, 변환된 학습 행의 불량만 복제한다. 복제는 새로운 불량 정보를 생성하지 않는다. 검증·보고 데이터는 복제하지 않는다. 오버샘플링과 최소 잎 표본 조건은 상호작용하므로 결과를 일반적인 불균형 처리 효과로 과도하게 일반화하지 않는다.

In [ ]:
def resample_indices(y,ratio,seed=SEED):
    y=np.asarray(y)
    idx=np.arange(len(y))
    if ratio is None:return idx
    positive=np.flatnonzero(y==1);negative=np.flatnonzero(y==0)
    assert len(positive)>0 and len(negative)>0
    needed=max(0,int(np.ceil(len(negative)*ratio))-len(positive))
    extra=np.random.default_rng(seed).choice(positive,size=needed,replace=True)
    return np.r_[idx,extra]

def fit_candidate(train,name):
    spec=SPECS[name]
    cols=[c for c in NUMERIC if train[c].nunique(dropna=True)>1]
    assert cols
    prep=ColumnTransformer([
        ('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),cols),
        ('part',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['PART_NAME'])])
    x=prep.fit_transform(train[cols+['PART_NAME']])
    y=train.defect.to_numpy()
    idx=resample_indices(y,spec['oversample_ratio'])
    assert np.all(idx<len(train))
    model=RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=5,
        class_weight=spec['class_weight'],random_state=SEED,n_jobs=2)
    model.fit(x[idx],y[idx])
    info={'train_original':len(y),'bad_original':int(y.sum()),'train_after':len(idx),
          'bad_after':int(y[idx].sum()),'numeric_features':len(cols)}
    return (prep,model,cols),info

def score_model(bundle,d):
    prep,model,cols=bundle
    return model.predict_proba(prep.transform(d[cols+['PART_NAME']]))[:,1]

cv_rows=[];oof_rows=[];started=time.perf_counter()
for name in SPECS:
    for fold,a,b in fold_data:
        model,info=fit_candidate(a,name)
        s=score_model(model,b)
        row={'model':name,'fold':fold,'AP':average_precision_score(b.defect,s),
             'ROC_AUC':roc_auc_score(b.defect,s),**info}
        cv_rows.append(row)
        oof_rows.append(pd.DataFrame({'model':name,'fold':fold,'record_id':b['_id'].to_numpy(),
                                     'y':b.defect.to_numpy(),'score':s}))
        print(f'{name} / {fold}: AP={row["AP"]:.4f}, 경과 {time.perf_counter()-started:.1f}초',flush=True)
        pd.DataFrame(cv_rows).to_csv(RESULT_DIR/'validation_folds.csv',index=False)
cv=pd.DataFrame(cv_rows);oof=pd.concat(oof_rows,ignore_index=True)
oof.to_csv(RESULT_DIR/'oof_predictions.csv',index=False)
assert len(cv)==12
pivot=cv.pivot(index='model',columns='fold',values='AP')
summary=pivot.copy()
summary['mean_AP']=pivot.mean(axis=1)
summary['min_AP']=pivot.min(axis=1)
summary['wins_vs_baseline']=(pivot.sub(pivot.loc[BASELINE],axis=1)>1e-12).sum(axis=1)
eligible=summary.loc[(summary.mean_AP>summary.loc[BASELINE,'mean_AP']+1e-12)&
                     (summary.wins_vs_baseline>=2)]
chosen=BASELINE if eligible.empty else eligible.sort_values(['mean_AP','min_AP'],ascending=False).index[0]
display(summary)
print('검증으로 선택:',chosen)
if chosen==BASELINE:print('안정적인 개선 조건을 만족한 후보가 없어 기준 모델을 유지합니다.')
summary.to_csv(RESULT_DIR/'validation_summary.csv')

## 6. 사용자 실행: 임계값 고정·동일 조건 최종 학습 — 최대 2회

각 모델의 검증 예측을 모아 임계값을 정한다. 서로 다른 시점에서 학습한 모델의 점수를 모으므로 최종 모델에서도 같은 검사 비율이 유지된다는 보장은 없다. 이 과정에서 기준 모델과 후보에 같은 규칙을 적용한다.

새 데이터에 대한 확정 검증이 아니라 이미 관측한 기간의 탐색 비교다. 판정 기준은 여기에서 저장하고 다음 셀의 보고 결과를 보고 바꾸지 않는다.

In [ ]:
def select_threshold(y,s):
    y=np.asarray(y);s=np.asarray(s)
    order=np.argsort(-s,kind='stable');sy=s[order];yy=y[order]
    ends=np.r_[np.flatnonzero(sy[:-1]!=sy[1:]),len(sy)-1]
    tp=np.cumsum(yy)[ends];inspections=ends+1;fp=inspections-tp;fn=y.sum()-tp
    table=pd.DataFrame({'threshold':sy[ends],'TP':tp,'FP':fp,'FN':fn,
        'inspection_rate':inspections/len(y),'F1':2*tp/(2*tp+fp+fn)})
    none=pd.DataFrame([{'threshold':np.inf,'TP':0,'FP':0,'FN':int(y.sum()),'inspection_rate':0.,'F1':0.}])
    table=pd.concat([table,none],ignore_index=True)
    best=table.loc[table.inspection_rate<=BUDGET].sort_values(['F1','FP','threshold'],ascending=[False,True,False]).iloc[0]
    return float(best.threshold),table

selected_names=list(dict.fromkeys([BASELINE,chosen]))
policies=[];final_models={}
for name in selected_names:
    v=oof.loc[oof.model==name]
    threshold,curve=select_threshold(v.y,v.score)
    policies.append({'model':name,'threshold':threshold,'selection':'temporal validation OOF only'})
    curve.to_csv(RESULT_DIR/f'{name}_validation_thresholds.csv',index=False)
policy=pd.DataFrame(policies)
policy.to_csv(RESULT_DIR/'frozen_thresholds.csv',index=False)
display(policy)
for name in selected_names:
    final_models[name],_=fit_candidate(refit,name)
    print('최종 학습 완료:',name,flush=True)

## 7. 같은 보고 데이터에서 성능·검사 부담 비교

선택된 임계값의 지표와 고정 10% 검사량에서의 탐지 수를 함께 본다. 후자는 운영 임계값의 성능이 아닌 고정 물량의 순위 진단이다. 10% 경계에서 같은 점수인 기록은 ID 순서로 정하며 동점 수를 출력한다. 같은 검사량에서도 이 동점 규칙에 따라 탐지 수가 달라질 수 있다.

평가 불량 8건으로 재현율은 한 건마다 12.5%p 변한다. 특히 RG3 LH 불량이 학습에 없다는 한계를 유지한다. 평가에서 높은 지표가 나왔다는 이유로 다른 모델·임계값을 사후 선택하지 않는다.

In [ ]:
def counts(y,p):
    y=np.asarray(y);p=np.asarray(p)
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'TP':int(tp),'FN':int(fn),'FP':int(fp),'TN':int(tn),
        'precision':float(tp/(tp+fp)) if tp+fp else 0.,
        'recall':float(tp/(tp+fn)) if tp+fn else np.nan,
        'F1':float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 0.,
        'inspections':int(tp+fp),'inspection_rate':float((tp+fp)/len(y)),
        'accuracy':float((tp+tn)/len(y))}

results=[];prediction_frames=[];by_part=[];fixed=[]
for name in selected_names:
    s=score_model(final_models[name],report)
    threshold=float(policy.loc[policy.model==name,'threshold'].iloc[0])
    pred=s>=threshold
    results.append({'model':name,'AP':average_precision_score(report.defect,s),
                    'ROC_AUC':roc_auc_score(report.defect,s),**counts(report.defect,pred)})
    p=pd.DataFrame({'record_id':report['_id'].to_numpy(),'time':report.time.to_numpy(),
        'product':report.PART_NAME.to_numpy(),'y':report.defect.to_numpy(),
        'score':s,'prediction':pred.astype(int),'model':name})
    prediction_frames.append(p)
    for part,g in p.groupby('product'):
        by_part.append({'model':name,'product':part,**counts(g.y,g.prediction)})
    k=int(np.floor(len(p)*BUDGET))
    ranking=p.sort_values(['score','record_id'],ascending=[False,True])
    top_ids=set(ranking.iloc[:k].record_id)
    boundary=float(ranking.iloc[k-1].score) if k else np.inf
    fixed.append({'model':name,'budget_k':k,'boundary_ties':int((p.score==boundary).sum()),
                  **counts(p.y,p.record_id.isin(top_ids))})
result=pd.DataFrame(results);part_result=pd.DataFrame(by_part)
display(result);display(pd.DataFrame(fixed));display(part_result)
result.to_csv(RESULT_DIR/'report_metrics.csv',index=False)
part_result.to_csv(RESULT_DIR/'report_by_product.csv',index=False)
pd.DataFrame(fixed).to_csv(RESULT_DIR/'fixed_budget_diagnostic.csv',index=False)
pd.concat(prediction_frames,ignore_index=True).to_csv(RESULT_DIR/'report_predictions.csv',index=False)
if chosen!=BASELINE:
    delta=result.set_index('model').loc[chosen,['AP','ROC_AUC','F1','recall','FP','inspections']]-result.set_index('model').loc[BASELINE,['AP','ROC_AUC','F1','recall','FP','inspections']]
    display(delta.to_frame('selected_minus_baseline'))
else:print('검증에서 개선 후보를 채택하지 않았습니다. 보고 결과로 선택을 바꾸지 않습니다.')
ax=result.set_index('model')[['precision','recall','F1']].plot.bar(rot=0,ylim=(0,1),figsize=(7,3.5))
ax.set_title('Previously observed November period: exploratory comparison')
plt.tight_layout();plt.show()

## 8. 결과 ZIP 받기 — 완료 후 이 ZIP과 실행 결과가 저장된 ipynb를 전달

In [ ]:
archive=RESULT_DIR/'imbalance_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(RESULT_DIR.iterdir()):
        if p.suffix in ['.csv','.json']:z.write(p,arcname=p.name)
print('완료:',archive)
print('성능 지표와 함께 미탐·검사량·구간별 일관성을 해석해야 합니다.')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))

## 출처·실험 이력

중소벤처기업부, Korea AI Manufacturing Platform(KAMP), 사출성형기 AI 데이터셋, KAIST(울산과학기술원, ㈜이피엠솔루션즈), 2020.12.14., https://www.kamp-ai.kr/

기준 모델 → 시계열 특성 비교 → RF 튜닝 → 제품별·준지도 실험의 한계 확인 → **원본/가공 데이터 관계 및 동일 공정값의 상반 라벨 확인 → 학습 구간 불균형 처리 비교**로 이어지는 후속 실험이다. 관측된 결과로 가설을 점검하고 효과가 없으면 기준 모델을 유지한다. 실행 전에는 개선 성과를 주장하지 않는다.

원자료와 제공 PDF는 저장소에 넣지 않는다. 결과 ZIP은 사용자와 분석을 위한 교환 파일이며 원본 ID별 예측을 포함한다. GitHub에는 프로젝트 설명·노트북·요약 결과만 정리한다.